# Framework Comparison — Fingerprint vs Descriptor

## Mission
Cross-validate the two independent clustering frameworks to identify high-confidence
synthesis candidates. A compound or cluster flagged by **both** frameworks represents
a region of chemical space that is simultaneously structurally distinct (fingerprint
framework) and physicochemically distinct (descriptor framework) — the strongest
possible justification for prioritizing it in synthesis.

## What this notebook asks
1. **Agreement**: Do the two frameworks cluster the same compounds together? (ARI)
2. **Synthesis overlap**: Do they recommend the same medoids as synthesis candidates?
3. **Disagreement analysis**: Where they disagree — structurally similar but
   physicochemically different compounds — these are the most scientifically interesting.
   They represent scaffold-hopping opportunities: same property profile, different structure.
4. **Combined confidence**: Compounds assigned to high-priority clusters by both
   frameworks = gold-standard synthesis targets.

## Inputs
- `results_fp/cluster_assignments_fp.csv` — from chemical_space_fingerprint.ipynb
- `results_desc/cluster_assignments_desc.csv` — from chemical_space_descriptors.ipynb
- `results_fp/synthesis_candidates_fp.csv`
- `results_desc/synthesis_candidates_desc.csv`

## Outputs
- `comparison/combined_confidence.csv` — compounds with dual-framework confidence score
- `comparison/disagreement_compounds.csv` — structurally similar but property-different
- 1 publication-ready figure: 2×2 UMAP panel showing both frameworks side by side

In [ ]:
!pip install -q umap-learn hdbscan scikit-learn-extra rdkit-pypi
print('Dependencies installed.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

FP_ASSIGNMENTS   = 'results_fp/cluster_assignments_fp.csv'
DESC_ASSIGNMENTS = 'results_desc/cluster_assignments_desc.csv'
FP_CANDIDATES    = 'results_fp/synthesis_candidates_fp.csv'
DESC_CANDIDATES  = 'results_desc/synthesis_candidates_desc.csv'

ACTIVITY_LABEL   = 'IC50 (nM)'
ACTIVITY_LOG     = True

# High-priority coverage flags from each framework
# Clusters with 'overlap' or 'library_only + high inter-cluster distance'
# are considered high-priority. Set to which coverage types to treat as high-priority:
HIGH_PRIORITY_COVERAGE = ['overlap', 'library_only']

OUTPUT_DIR = 'comparison'
# ═══════════════════════════════════════════════════════════════════════════════
print('Configuration loaded.')

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.metrics import adjusted_rand_score
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, DataStructs
from IPython.display import display

warnings.filterwarnings('ignore')
Path(OUTPUT_DIR).mkdir(exist_ok=True)
print('Imports OK.')

## 1. Load Both Framework Outputs

In [ ]:
fp_df   = pd.read_csv(FP_ASSIGNMENTS)
desc_df = pd.read_csv(DESC_ASSIGNMENTS)
fp_cand   = pd.read_csv(FP_CANDIDATES)
desc_cand = pd.read_csv(DESC_CANDIDATES)

print(f'Fingerprint framework : {len(fp_df):,} compounds, {fp_df["km_cluster"].nunique()} K-Medoid clusters')
print(f'Descriptor framework  : {len(desc_df):,} compounds, {desc_df["km_cluster"].nunique()} K-Medoid clusters')

# Merge on SMILES (the common compound identifier)
merged = pd.merge(
    fp_df.rename(columns={'km_cluster':'km_fp','hdb_cluster':'hdb_fp',
                          'umap_x':'umap_x_fp','umap_y':'umap_y_fp'}),
    desc_df.rename(columns={'km_cluster':'km_desc','hdb_cluster':'hdb_desc',
                            'umap_x':'umap_x_desc','umap_y':'umap_y_desc',
                            'pca_x':'pca_x_desc','pca_y':'pca_y_desc'}),
    on=['smiles','name','source'],
    how='inner',
    suffixes=('_fp','_desc'),
)
print(f'\nOverlapping compounds (in both frameworks): {len(merged):,}')
print(f'Compounds only in FP framework  : {len(fp_df) - len(merged):,}')
print(f'Compounds only in DESC framework: {len(desc_df) - len(merged):,}')
print('Note: discrepancies arise from different deduplication outcomes in each framework.')

## 2. Framework Agreement — Adjusted Rand Index

ARI measures how similarly the two frameworks partition the same compounds.
- ARI = 1.0: identical cluster assignments
- ARI = 0.0: random (no agreement)
- ARI < 0: worse than random

**Interpretation for library design:**
- High ARI (> 0.5): structural similarity and physicochemical similarity are
  correlated in this library — the two frameworks tell the same story.
  Either framework alone is sufficient for synthesis decisions.
- Low ARI (< 0.3): the library has structurally diverse compounds with similar
  properties (scaffold hops) — **both frameworks are needed** to fully characterize
  the space. The disagreement regions are scientifically interesting.

In [ ]:
# ARI between K-Medoids frameworks
ari_km = adjusted_rand_score(merged['km_fp'], merged['km_desc'])
# ARI between HDBSCAN frameworks (exclude noise from both)
non_noise = (merged['hdb_fp'] != -1) & (merged['hdb_desc'] != -1)
if non_noise.sum() > 20:
    ari_hdb = adjusted_rand_score(merged.loc[non_noise,'hdb_fp'], merged.loc[non_noise,'hdb_desc'])
else:
    ari_hdb = np.nan

print('Framework Agreement (Adjusted Rand Index)')
print(f'  K-Medoids ARI  : {ari_km:.4f}')
print(f'  HDBSCAN ARI    : {ari_hdb:.4f}' if not np.isnan(ari_hdb) else '  HDBSCAN ARI: insufficient non-noise points')
print()
if ari_km > 0.5:
    print('  Interpretation: HIGH agreement — structural and physicochemical space')
    print('  are correlated. Both frameworks support the same synthesis priorities.')
elif ari_km > 0.3:
    print('  Interpretation: MODERATE agreement — use both frameworks.')
    print('  Disagreement regions contain scaffold-hopping candidates.')
else:
    print('  Interpretation: LOW agreement — frameworks reveal complementary structure.')
    print('  Library has structurally diverse compounds with similar properties.')
    print('  Disagreement analysis (Section 4) will be the most valuable output.')

## 3. Synthesis Candidate Overlap

Do the two frameworks recommend the same medoids for synthesis?
Overlap is measured by Tanimoto similarity between medoid pairs across frameworks.

In [ ]:
def smiles_to_fp(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048) if mol else None

fp_medoid_fps   = [(row['cluster'], row['medoid_smiles'], smiles_to_fp(row['medoid_smiles']))
                   for _, row in fp_cand.iterrows()]
desc_medoid_fps = [(row['cluster'], row['medoid_smiles'], smiles_to_fp(row['medoid_smiles']))
                   for _, row in desc_cand.iterrows()]

# Cross-framework Tanimoto similarity between all medoid pairs
overlap_rows = []
for fp_clust, fp_smi, fp_fp in fp_medoid_fps:
    for desc_clust, desc_smi, desc_fp in desc_medoid_fps:
        if fp_fp is None or desc_fp is None:
            continue
        sim = DataStructs.TanimotoSimilarity(fp_fp, desc_fp)
        overlap_rows.append({
            'fp_cluster':   fp_clust,
            'desc_cluster': desc_clust,
            'fp_smiles':    fp_smi,
            'desc_smiles':  desc_smi,
            'tanimoto':     round(sim, 3),
        })

overlap_df = pd.DataFrame(overlap_rows)

# Best match per FP medoid
best_match = overlap_df.loc[overlap_df.groupby('fp_cluster')['tanimoto'].idxmax()]
print('Best Tanimoto match for each FP-framework medoid in DESC-framework medoids:')
display(best_match[['fp_cluster','desc_cluster','tanimoto']].to_string(index=False))

n_high_overlap = (best_match['tanimoto'] > 0.5).sum()
print(f'\nMedoids with Tanimoto > 0.5 match across frameworks: {n_high_overlap}/{len(best_match)}')
print('These represent double-validated synthesis candidates.')

overlap_df.to_csv(f'{OUTPUT_DIR}/medoid_cross_framework_similarity.csv', index=False)

## 4. Disagreement Analysis — Scaffold Hops

Compounds where the frameworks disagree are the most scientifically interesting.

**Type 1 — Same structure, different properties** (FP agrees, DESC disagrees):
Structurally similar compounds that differ in physicochemical properties.
These represent subtle property modifications on the same scaffold — SAR-relevant.

**Type 2 — Different structure, same properties** (FP disagrees, DESC agrees):
Structurally diverse compounds with similar physicochemical profiles.
These are scaffold hops — different structures that occupy the same property space.
High value for library design: chemically distinct routes to the same target property profile.

In [ ]:
# Compound-level disagreement: assigned to same K-Medoids cluster in one framework,
# different cluster in the other

# For each compound, compute:
# - fp_cluster_priority: is this compound in a high-priority FP cluster?
# - desc_cluster_priority: is this compound in a high-priority DESC cluster?

fp_priority_clusters   = fp_cand[fp_cand['coverage'].isin(HIGH_PRIORITY_COVERAGE)]['cluster'].tolist()
desc_priority_clusters = desc_cand[desc_cand['coverage'].isin(HIGH_PRIORITY_COVERAGE)]['cluster'].tolist()

merged['fp_priority']   = merged['km_fp'].isin(fp_priority_clusters)
merged['desc_priority'] = merged['km_desc'].isin(desc_priority_clusters)

# Confidence categories
merged['confidence'] = 'low'
merged.loc[merged['fp_priority'] & merged['desc_priority'],  'confidence'] = 'high'
merged.loc[merged['fp_priority'] ^ merged['desc_priority'],  'confidence'] = 'medium'

conf_counts = merged['confidence'].value_counts()
print('Compound confidence distribution:')
for level in ['high','medium','low']:
    n = conf_counts.get(level, 0)
    print(f'  {level:6s}: {n:,} ({100*n/len(merged):.1f}%)')

print('\nHigh confidence = flagged by BOTH frameworks as high-priority')
print('Medium confidence = flagged by ONE framework only (disagreement region)')
print('Low confidence = not prioritized by either framework')

# Scaffold hop candidates: medium confidence compounds that FP framework
# did NOT flag but DESC did (same property profile, different structure)
scaffold_hops = merged[(merged['desc_priority']) & (~merged['fp_priority'])].copy()
print(f'\nScaffold hop candidates (DESC-priority only, structurally distinct): {len(scaffold_hops):,}')

merged.to_csv(f'{OUTPUT_DIR}/combined_confidence.csv', index=False)
scaffold_hops[['name','smiles','source','km_fp','km_desc']].to_csv(
    f'{OUTPUT_DIR}/disagreement_compounds.csv', index=False)

## 5. Publication Figure — 2×2 Framework Comparison

Four-panel figure showing both UMAP canvases side by side, colored by cluster labels
from each respective framework:
- Top row: fingerprint framework UMAP, colored by (1) FP clusters, (2) DESC clusters
- Bottom row: descriptor framework UMAP, colored by (3) FP clusters, (4) DESC clusters

If the coloring is consistent across rows → high ARI, frameworks agree.
If coloring is inconsistent → low ARI, frameworks capture different information.

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)
cmap10 = plt.cm.tab10
cmap20 = plt.cm.tab20

panels = [
    (gs[0,0], 'umap_x_fp',   'umap_y_fp',   'km_fp',
     'FP-UMAP canvas | colored by FP K-Medoids clusters'),
    (gs[0,1], 'umap_x_fp',   'umap_y_fp',   'km_desc',
     'FP-UMAP canvas | colored by DESC K-Medoids clusters\n(consistency check — do DESC clusters align with FP topology?)'),
    (gs[1,0], 'umap_x_desc', 'umap_y_desc', 'km_fp',
     'DESC-UMAP canvas | colored by FP K-Medoids clusters\n(consistency check — do FP clusters align with DESC topology?)'),
    (gs[1,1], 'umap_x_desc', 'umap_y_desc', 'km_desc',
     'DESC-UMAP canvas | colored by DESC K-Medoids clusters'),
]

for subplot_spec, x_col, y_col, color_col, title in panels:
    ax = fig.add_subplot(subplot_spec)
    labs = sorted(merged[color_col].unique())
    for lab in labs:
        mask = merged[color_col] == lab
        ax.scatter(merged.loc[mask, x_col], merged.loc[mask, y_col],
                   c=[cmap10(int(lab)%10)], s=5, alpha=0.4,
                   label=f'K{lab}', rasterized=True)

    # Mark high-confidence compounds
    high_conf = merged['confidence'] == 'high'
    if high_conf.sum():
        ax.scatter(merged.loc[high_conf, x_col], merged.loc[high_conf, y_col],
                   s=25, marker='o', facecolors='none', edgecolors='gold',
                   linewidths=0.6, alpha=0.6, zorder=5, label='High confidence')

    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel(f'{x_col.split("_")[1].upper()} 1')
    ax.set_ylabel(f'{x_col.split("_")[1].upper()} 2')
    ax.legend(fontsize=6, ncol=3, loc='upper right')

fig.suptitle(
    f'Framework Comparison: Fingerprint vs Descriptor\n'
    f'K-Medoids ARI = {ari_km:.3f}  |  '
    f'High-confidence candidates = {(merged["confidence"]=="high").sum():,}',
    fontsize=12, fontweight='bold'
)
plt.savefig(f'{OUTPUT_DIR}/publication_figure_framework_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Publication figure saved.')

## 6. Final Synthesis Recommendation

In [ ]:
# High confidence medoids: appear in both synthesis candidate lists with high overlap
double_validated = best_match[best_match['tanimoto'] > 0.5]

print('=' * 65)
print('SYNTHESIS RECOMMENDATION SUMMARY')
print('=' * 65)
print(f'Total compounds analyzed        : {len(merged):,}')
print(f'K-Medoids ARI (FP vs DESC)      : {ari_km:.4f}')
print(f'Framework agreement             : {"HIGH" if ari_km>0.5 else "MODERATE" if ari_km>0.3 else "LOW"}')
print()
print(f'High-confidence candidates      : {(merged["confidence"]=="high").sum():,}')
print(f'  (flagged by both frameworks as high-priority)')
print(f'Scaffold hop candidates         : {len(scaffold_hops):,}')
print(f'  (DESC-priority, FP-diverse — different scaffold, same property profile)')
print(f'Double-validated medoids        : {len(double_validated)}/{len(fp_cand)}')
print(f'  (Tanimoto > 0.5 match between FP and DESC medoids)')
print()
print('Recommended synthesis priority:')
print('  1. Double-validated medoids (both frameworks agree on structure AND property)')
print('  2. High-confidence compounds in overlap clusters (literature-validated space)')
print('  3. Scaffold hop candidates (novel structural access to target property profile)')
print('=' * 65)

if len(double_validated):
    print('\nDouble-validated medoid SMILES:')
    for _, row in double_validated.iterrows():
        print(f'  FP K{row["fp_cluster"]} ↔ DESC K{row["desc_cluster"]}  Tan={row["tanimoto"]:.3f}  {row["fp_smiles"][:60]}...')

In [ ]:
# Draw double-validated medoids as structure grid
if len(double_validated):
    mols, legends = [], []
    for _, row in double_validated.iterrows():
        mol = Chem.MolFromSmiles(str(row['fp_smiles']))
        if mol:
            from rdkit.Chem import AllChem
            AllChem.Compute2DCoords(mol)
            mols.append(mol)
            legends.append(f'FP K{row["fp_cluster"]} ↔ DESC K{row["desc_cluster"]}\nTan={row["tanimoto"]:.3f}')
    if mols:
        img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(300, 250),
                                   legends=legends, returnPNG=False)
        img.save(f'{OUTPUT_DIR}/double_validated_medoids.png')
        display(img)
        print('Double-validated synthesis candidates structure grid saved.')
else:
    print('No double-validated medoids (Tanimoto > 0.5 threshold). '
          'Frameworks are divergent — both are needed. Inspect scaffold hops instead.')